# BERT Gender Classifier — Blog Authorship Corpus

Trains `bert-base-uncased` on the Blog Authorship Corpus (Schler et al., 2006) — 19,320 English bloggers with gender labels.
Evaluates cross-dataset on LiLaH EN (Facebook hate groups).

This is a much larger training set than PAN14 (~420 authors) and should give stronger gender predictions.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or better)
2. Run `extract_blog_corpus.py` locally to produce `blog_gender.csv`
3. Upload `blog_gender.csv` to `My Drive/thesis/`
4. Upload `hate_speech_only.tsv` to `My Drive/thesis/` (LiLaH EN, if not already there)
5. Run all cells top to bottom

Model saves to `My Drive/thesis/bert_blog_gender_model/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install -q -U transformers accelerate
import transformers
print(f'transformers: {transformers.__version__}')

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────
MODEL_NAME = 'bert-base-uncased'

TRAIN_CSV  = '/content/drive/MyDrive/thesis/blog_gender.csv'
LILAH_TSV  = '/content/drive/MyDrive/thesis/hate_speech_only.tsv'
OUTPUT_DIR = '/content/drive/MyDrive/thesis/bert_blog_gender_model'

SAMPLE_PER_CLASS = 50000   # 50k M + 50k F = 100k total

MAX_LENGTH       = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE  = 32
GRAD_ACCUM_STEPS = 2       # effective batch = 32
NUM_EPOCHS       = 3
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
RANDOM_STATE     = 42
LABELS           = ['F', 'M']
# ───────────────────────────────────────────────────────────────────────────
print(f'Model:  {MODEL_NAME}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    precision_recall_fscore_support
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors=None
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1, 'macro_precision': p, 'macro_recall': r}

print('Imports OK')

In [ ]:
# Load Blog Corpus
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=['gender', 'text']).copy()
df['text']   = df['text'].astype(str).str[:3000]   # cap very long blog posts
df['gender'] = df['gender'].astype(str).str.strip()
df = df[df['gender'].isin(LABELS)].reset_index(drop=True)

print(f'Full corpus: {len(df)} bloggers')
print(df['gender'].value_counts())
print(f'Avg text length: {df["text"].str.len().mean():.0f} chars')

# Balanced sample
sampled = (
    df.groupby('gender', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CLASS), random_state=RANDOM_STATE))
)
sampled = sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'\nTraining sample: {len(sampled)} texts')
print(sampled['gender'].value_counts())

In [ ]:
# Label encoding + train/val split
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label  = {idx: label for label, idx in label2id.items()}
sampled['label'] = sampled['gender'].map(label2id)

train_df, val_df = train_test_split(
    sampled, test_size=0.1, random_state=RANDOM_STATE, stratify=sampled['label']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'task': 'gender', 'label2id': label2id,
               'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2)
print('Label mapping saved')

In [ ]:
# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizing training set...')
train_dataset = TextDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
print('Tokenizing validation set...')
val_dataset   = TextDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer, MAX_LENGTH)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
# Load BERT
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to='none',
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
print(f'Done in {round((time.time()-start)/60, 1)} min')

In [ ]:
# Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')

In [ ]:
# In-domain validation (Blog Corpus)
preds_out = trainer.predict(val_dataset)
pred_ids  = preds_out.predictions.argmax(axis=1)
true_ids  = preds_out.label_ids
print('=== In-domain validation (Blog Corpus) ===')
print(f'Macro F1: {f1_score(true_ids, pred_ids, average="macro", zero_division=0):.4f}')
print(classification_report(true_ids, pred_ids, target_names=LABELS, zero_division=0))

## Cross-dataset evaluation — LiLaH EN

Trained on Blog Corpus (blog posts), tested on LiLaH EN (Facebook hate groups).  
Compare with PAN14-trained BERT (macro F1 = 0.487 on LiLaH EN).

In [ ]:
# Load LiLaH EN
lilah = pd.read_csv(LILAH_TSV, sep='\t')
lilah = lilah.dropna(subset=['text', 'gender']).copy()
lilah['text'] = lilah['text'].astype(str)
lilah['gender'] = (
    lilah['gender'].astype(str).str.strip()
    .map(lambda x: {'male':'M','female':'F','m':'M','f':'F'}.get(x.lower(), x.upper()))
)
lilah = lilah[lilah['gender'].isin(LABELS)].reset_index(drop=True)
print(f'LiLaH EN: {len(lilah)} texts')
print(lilah['gender'].value_counts())

In [ ]:
# Predict on LiLaH EN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
model.to(device)

BATCH = 64
all_preds = []
for i in range(0, len(lilah), BATCH):
    batch = lilah['text'].tolist()[i:i+BATCH]
    enc = tokenizer(batch, truncation=True, padding=True,
                    max_length=MAX_LENGTH, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())

pred_labels = [id2label[i] for i in all_preds]
true_labels = lilah['gender'].tolist()

print('=== Cross-dataset: LiLaH EN ===')
mac = f1_score(true_labels, pred_labels, average='macro', labels=LABELS, zero_division=0)
per = f1_score(true_labels, pred_labels, average=None,   labels=LABELS, zero_division=0)
print(f'Macro F1: {mac:.4f}  |  F1-F: {per[0]:.4f}  F1-M: {per[1]:.4f}')
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))

print('\n=== Comparison ===')
print(f'  PAN14 BERT (420 authors):  Macro F1 = 0.487  F1-F=0.292  F1-M=0.681')
print(f'  Blog BERT (~19k authors):  Macro F1 = {mac:.3f}  F1-F={per[0]:.3f}  F1-M={per[1]:.3f}')

# Save predictions
out_df = lilah.copy()
out_df['predicted'] = pred_labels
out_path = os.path.join(OUTPUT_DIR, 'bert_blog_gender_lilah_predictions.csv')
out_df.to_csv(out_path, index=False)
print(f'\nPredictions saved to: {out_path}')